# LensV3 Question Data Processing Pipeline (LLM version)

Same goal as `process.ipynb`, but one Gemini call per session replaces langdetect, Google Translate,
sentence-transformers, KMeans and the rule-based reframing.

**Steps:**
1. Batch the data by `session_id`
2. For each session, ask Gemini to translate, filter non-questions, group into up to 6 topics, label them, and reframe each question
3. Output JSON per session with original and reframed questions per topic

Requires `GEMINI_API_KEY` in your environment (`pip install google-genai pandas`).

## Step 0: Imports & Configuration

In [ ]:
import json
from pathlib import Path

import pandas as pd
from google import genai
from google.genai import types
from pydantic import BaseModel, Field

INPUT_DIR  = Path('input')
OUTPUT_DIR = Path('output')
OUTPUT_DIR.mkdir(exist_ok=True)

CSV_FILE   = INPUT_DIR / 'lensV3_questions_data - Dasra (1).csv'
NUM_TOPICS = 6
MODEL      = 'gemini-3.8-flash'

client = genai.Client()  # reads GEMINI_API_KEY
print('Setup complete ✓')

Setup complete ✓


## Step 1: Load & Batch the Data by `session_id`

In [ ]:
df = pd.read_csv(CSV_FILE)
df.columns = df.columns.str.strip()
df['event_time'] = pd.to_datetime(df['event_time'])
df = df.sort_values(['session_id', 'event_time']).reset_index(drop=True)

session_groups = dict(tuple(df.groupby('session_id')))

print(f'Loaded {len(df)} rows across {len(session_groups)} sessions')
for sid, group in session_groups.items():
    print(f'  Session {sid}: {len(group)} entries')

Loaded 37 rows across 1 sessions
  Session 41b0dbe7-48b2-46ff-85e2-1442d1a53d10: 37 entries


## Step 2: Define the Structured Output & Prompt

Gemini refers to questions **by ID only** and never rewrites the originals. Originals are always copied from the CSV,
so they can't be altered by the model.

In [ ]:
class QuestionResult(BaseModel):
    id: int = Field(description='The ID of the input entry')
    is_question: bool = Field(description='False for commands/acknowledgements like "Yes", "Approved plan", "Move to the next step"')
    english: str = Field(description='The entry translated to English (unchanged if already English)')
    reframed: str = Field(description='A clear, concise, well-formed question in English. Empty string if is_question is false')


class Topic(BaseModel):
    label: str = Field(description='Topic title of 1-2 words, e.g. "Funding Models" or "Agriculture"')
    question_ids: list[int] = Field(description='IDs of the questions in this topic')


class SessionResult(BaseModel):
    questions: list[QuestionResult]
    topics: list[Topic]


SYSTEM_PROMPT = f"""You analyse the questions a user asked in one session of a research assistant about philanthropy and the social sector.

You receive numbered entries. Some are in languages other than English, and some are not questions at all
(commands or acknowledgements such as "Yes", "Approved plan", "Move to the next step", "Mark this complete").

For each entry:
- Translate it to English (keep it unchanged if it is already English).
- Decide whether it is a meaningful question or request for information (is_question).
- If it is, reframe it as a clear, concise question that keeps the original intent and specifics
  (names, places, numbers). Fix spelling. Turn instructions such as "Compare X with Y" into direct
  questions ("How does X compare with Y?"), not generic wrappers.

Then group only the meaningful questions into at most {NUM_TOPICS} topics by what they are about, and give each
topic a short title of 1 to 2 words (e.g. "Funding Models", "Agriculture"). Use fewer topics if the questions don't naturally split into {NUM_TOPICS}.
Every meaningful question must appear in exactly one topic. Return an entry in `questions` for every input ID."""

## Step 3: Process Each Session with Gemini

In [ ]:
def analyse_session(texts: list[str]) -> SessionResult:
    entries = '\n'.join(f'[{i}] {t}' for i, t in enumerate(texts))
    response = client.models.generate_content(
        model=MODEL,
        contents=f'Session entries:\n\n{entries}',
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_PROMPT,
            response_mime_type='application/json',
            response_schema=SessionResult,
            temperature=0,
        ),
    )
    if response.parsed is None:
        finish = response.candidates[0].finish_reason if response.candidates else response.prompt_feedback
        raise RuntimeError(f'No valid structured output (finish reason: {finish})')
    return response.parsed


results = {}
for sid, group in session_groups.items():
    texts = [str(t).strip() for t in group['question_text']]
    print(f'Processing session {sid} ({len(texts)} entries)...')
    results[sid] = (texts, analyse_session(texts))
print('Done ✓')

Processing session 41b0dbe7-48b2-46ff-85e2-1442d1a53d10 (37 entries)...
Done ✓


## Step 4: Validate & Build Output JSON

In [ ]:
output = []

for sid, (texts, result) in results.items():
    by_id = {q.id: q for q in result.questions if 0 <= q.id < len(texts)}
    kept_ids = {i for i, q in by_id.items() if q.is_question}

    # Guard against the model dropping or duplicating question IDs across topics
    assigned = set()
    topics = []
    for topic in result.topics[:NUM_TOPICS]:
        ids = [i for i in topic.question_ids if i in kept_ids and i not in assigned]
        assigned.update(ids)
        if ids:
            topics.append((topic.label, ids))
    unassigned = sorted(kept_ids - assigned)
    if unassigned:
        print(f'  ⚠ {len(unassigned)} question(s) not placed in a topic, adding to "Other"')
        topics.append(('Other', unassigned))

    session_out = {'session_id': sid}
    for n, (label, ids) in enumerate(topics, start=1):
        session_out[f'topic{n}'] = {
            'topic_label': label,
            'set_of_original_questions': [texts[i] for i in ids],
            'set_of_reframed_questions': [by_id[i].reframed or by_id[i].english for i in ids],
        }
    output.append(session_out)

    dropped = [texts[i] for i in range(len(texts)) if i not in kept_ids]
    print(f'Session {sid}: {len(kept_ids)} questions kept, {len(dropped)} filtered out, {len(topics)} topics')
    for label, ids in topics:
        print(f'    {label}: {len(ids)}')

print(json.dumps(output, indent=2, ensure_ascii=False))

Session 41b0dbe7-48b2-46ff-85e2-1442d1a53d10: 17 questions kept, 20 filtered out, 5 topics
    Enterprise Strategy: 1
    Agriculture: 4
    Philanthropic Intermediaries: 6
    Concessional Capital: 3
    Report Analysis: 3
[
  {
    "session_id": "41b0dbe7-48b2-46ff-85e2-1442d1a53d10",
    "topic1": {
      "topic_label": "Enterprise Strategy",
      "set_of_original_questions": [
        "How does the Foundation define an \"invention-based enterprise,\" and why is that the right unit of focus compared with startups generally?"
      ],
      "set_of_reframed_questions": [
        "How does the Foundation define an \"invention-based enterprise,\" and why is that the right unit of focus compared with startups generally?"
      ]
    },
    "topic2": {
      "topic_label": "Agriculture",
      "set_of_original_questions": [
        "Why did horticultre farmer joined raitha sangha",
        "ತೋಟಗಾರಿಕೆ ರೈತರು ರೈತ ಸಂಘವನ್ನು ಏಕೆ ಸೇರಿದರು?",
        "ತೋಟಗಾರಿಕೆ ರೈತರು, ಅ, ರೈತ ಸಂಘವನ್ನು ಸೇರಲು ಕಾರಣವ

In [ ]:
output_path = OUTPUT_DIR / 'output_llm.json'
with open(output_path, 'w', encoding='utf-8') as f:
    json.dump(output, f, indent=2, ensure_ascii=False)
print(f'Output saved to: {output_path}')

Output saved to: output/output_llm.json
